In [3]:
import tkinter as tk
from tkinter import *
from tkinter import messagebox
import os
from subprocess import Popen
import time


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image_dataset_from_directory
from cv2 import *
from keras.preprocessing.image import ImageDataGenerator

#Create the window
window = tk.Tk()
window.title("Drowsiness Detection System")
window.geometry('350x270')

#Variable that tracks if the start button has been clicked
running = True

def scanning():
    if running:  # Only do this if the Stop button has not been clicked
        
        #Call TiredTest() to test the face
        eyeResult, netResult = TiredTest()
        
        print(str(eyeResult) + " " + str(netResult))  #Debugging

        #If both tests return 1, the person is very tired
        if(netResult and eyeResult == 1):
            messagebox.showwarning("Warning", "You look very tired.  30 minute break recommended.")
        #If one test returns 1, the person is somewhat tired
        elif (netResult or eyeResult == 1):
            messagebox.showwarning("Warning", "You look a little tired.  10 minute break recommended.")
        #The person is not tired
        else:
            messagebox.showinfo("Info", "No break needed")
    
    
    #Wait for set time before repeating
    #Currently set to 5 seconds for testing
    window.after(5000, scanning) #30 minutes = 1800000

#When start button clicked
def start():
    #Make sure running is true
    global running
    running = True

#When stop button clicked
def exit():
    #Set running to false to stop the detection loop
    global running
    running = False

    window.destroy()
    


#Create the buttons
startButton = tk.Button(window, height = 1, width = 15, text = "Start", command = start)
exitButton = tk.Button(window, height = 1, width = 15, text = "Exit", command = exit)


#Make the program wait before scanning again
#If the statement is not here also, the start button will not start the program.
window.after(5000, scanning)  # After 30 minutes, call scanning



startButton.pack()
exitButton.pack()


window.mainloop()

Instructions for updating:
Please use instead:* `np.argmax(model.predict(x), axis=-1)`,   if your model does multi-class classification   (e.g. if it uses a `softmax` last-layer activation).* `(model.predict(x) > 0.5).astype("int32")`,   if your model does binary classification   (e.g. if it uses a `sigmoid` last-layer activation).
0 0
1 0
1 0
1 0


In [2]:
def TiredTest():
    #Capture a 5 second video using the webcam and save it

    import cv2
    import time
    import os
    from keras.models import load_model
    import numpy as np

    #For eye detection
    face = cv2.CascadeClassifier('haar cascade files\haarcascade_frontalface_alt.xml')
    leye = cv2.CascadeClassifier('haar cascade files\haarcascade_lefteye_2splits.xml')
    reye = cv2.CascadeClassifier('haar cascade files\haarcascade_righteye_2splits.xml')

    #Load the model
    model = load_model('custmodel.h5')
    path = os.getcwd()
    cap = cv2.VideoCapture(0)
    count=0
    score=0
    rpred=[99]
    lpred=[99]

    #Eye detection result - will be 0 for not tired or 1 for tired
    eye_detection_result = 0

    # The duration in seconds of the video captured
    capture_duration = 5
    i = 0

    #Capture from the webcam
    cap = cv2.VideoCapture(0)

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('test.mp4',fourcc, 20.0, (640,480))

    start_time = time.time()

    while( int(time.time() - start_time) < capture_duration ):  #For duration of video capture
        ret, frame = cap.read()
        cv2.imwrite("TestImages/" + str(i)+'.jpg', frame)  #Save each individual frame as image
        i = i + 1

        #Eye Detection
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        #Detect the face and the eyes
        faces = face.detectMultiScale(gray,minNeighbors=5,scaleFactor=1.1,minSize=(25,25))
        left_eye = leye.detectMultiScale(gray)
        right_eye =  reye.detectMultiScale(gray)


        for (x,y,w,h) in faces:
                cv2.rectangle(frame, (x,y) , (x+w,y+h) , (100,100,100) , 1 )
                
        #Detect the right eye
        for (x,y,w,h) in right_eye:
            r_eye=frame[y:y+h,x:x+w]
            count=count+1
            r_eye = cv2.cvtColor(r_eye,cv2.COLOR_BGR2GRAY)
            r_eye = cv2.resize(r_eye,(24,24))
            r_eye= r_eye/255
            r_eye=  r_eye.reshape(24,24,-1)
            r_eye = np.expand_dims(r_eye,axis=0)
            #Make a prediction
            rpred = model.predict_classes(r_eye)
            if(rpred[0]==1):
                lbl='Open' 
            if(rpred[0]==0):
                lbl='Closed'
            break

        #detect the left eye
        for (x,y,w,h) in left_eye:
            l_eye=frame[y:y+h,x:x+w]
            count=count+1
            l_eye = cv2.cvtColor(l_eye,cv2.COLOR_BGR2GRAY)  
            l_eye = cv2.resize(l_eye,(24,24))
            l_eye= l_eye/255
            l_eye=l_eye.reshape(24,24,-1)
            l_eye = np.expand_dims(l_eye,axis=0)
            #Make the prediction
            lpred = model.predict_classes(l_eye)
            if(lpred[0]==1):
                lbl='Open'   
            if(lpred[0]==0):
                lbl='Closed'
            break

        #If both eyes are closed, increase the score
        if(rpred[0]==0 and lpred[0]==0):
            score=score+1

        # if one or both eyes are open, decrease the score
        else:
            score=score-1


        if(score<0):
            score=0   

        if(score>15):  #Eyes are closed for a long time, so they are tired
            eye_detection_result = 1
        else:
            eye_detection_result = 0  # eyes were open, return 0 for not tired


    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    
    
    
    #Import the CNN model
    model = tf.keras.models.load_model('model')
    
    
    
    
    
    #Test each frame of the video
    #Make list of classification for each frame
    #Average the classification and make a final prediction for the whole video
    import os
    from pathlib import Path
    import numpy as np
    #import matplotlib.pyplot as plt

    notTiredCount = 0 #number of frames predicted as not tired
    tiredCount = 0 # number of frames predicted as tired
    total = 0 #total number of frames

    folder_dir = 'TestImages'
    frameList = []  # holds the images


    images = Path(folder_dir).glob('*.jpg')  # Path to each frame of video as images
    for image in images:       #Add each image to a list
        frameList.append(image)

    image_size = (256, 256)
    # loop through the list of images





    for i in range(len(frameList)):
         # Load the current image and make it into an array
        img = keras.preprocessing.image.load_img(
        frameList[i], target_size=image_size
        )
        img_array = keras.preprocessing.image.img_to_array(img)
        # Expand the dims to fit
        img_array = tf.expand_dims(img_array, 0) 

        # Make a prediction using the image
        classPredict = (model.predict(img_array)> 0.5).astype("int32")
        # Increase the tired or not tired count based on the prediction
        if(classPredict == 0):
            notTiredCount+=1
        else:
            tiredCount+=1
        # Add to the total number of frames
        total+=1


    neural_network_result = 0 #Variable to store result

    #If there are more not tired frames than tired, predict not tired.
    #If there are more tired frames than not tired, predict tired.
    if notTiredCount > tiredCount:
        neural_network_result = 0
    elif tiredCount > notTiredCount:
        neural_network_result = 1
    else:
        #print("Error")
        pass
        
        
        
        
    #Delete all images in the TestImages folder.
    #Prevents leftover images being used in the next
    #round of detection

    import os, shutil
    folder = 'TestImages'
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))
            
    #Return the results of both tests       
    return eye_detection_result, neural_network_result
            
